In [4]:
from typing import Literal, TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain.tools import tool
from IPython.display import Image, display
from langgraph.prebuilt import ToolNode, tools_condition
from langchain.agents import create_agent
import os
load_dotenv()
qwen_model = init_chat_model("openai:qwen3.7-plus", temperature=0, base_url=os.getenv("ROOT_URL"))


In [ ]:
# 父 State
class IntentState(MessagesState):
    intent: str

# 子 State
class WeatherState(MessagesState):
    foo: str # 故意展示与父不同的字段

In [ ]:
# ====== Step 1: 准备工具 ======
# 工具
@tool
def get_weather(city: str) -> str:
    """获取城市天气"""
    weather_data = {
        "北京": "晴天 25度",
        "上海": "多云 28度",
        "杭州": "小雨 22度",
    }
    return weather_data.get(city, f"未找到{city}的天气信息")

# 组成工具数组
tools = [get_weather]

# ====== Step 2: 准备模型 ======
# 定义模型
# ====== Step 3: 定义Agent ======
weather_agent = create_agent(qwen_model, tools=[get_weather], state_schema="WeatherState")

In [ ]:
from typing import Literal

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END, MessagesState

# ===============Step 1: 初始化chat模型===============
llm = qwen_model

# ===============Step 2: 定义Node===============
def intent_node(state: IntentState):
    intent = llm.invoke(f'''
    根据用户的请求判断用户意图，可选意图：weather、translate或chat。
    user_query: {state['messages'][-1].content}
    ''')
    return {'intent': intent.text}

# 不需要手动调用了，所以把handle_weather节点注释掉
# def handle_weather(state: IntentState):
#     return {'messages': [AIMessage(content='晴 25度')]}

def translate_node(state: IntentState):
    result = llm.invoke(f'''
    按照用户要求翻译，只输出翻译结果，不要任何解释。
    user_query:{state['messages'][-1].content}
    ''')
    return {'messages': [result]}

def chat_node(state: IntentState):
    result = llm.invoke(state['messages'])
    return {'messages': [result]}

def intent_router(state: IntentState) -> Literal['weather', 'translate', 'chat']:
    '''根据意图路由到不同处理器'''
    return state['intent']

# ===============Step 5: 创建Graph===============
builder = StateGraph(IntentState)
builder.add_node('intent', intent_node)
builder.add_node('weather', weather_agent)
builder.add_node('chat', chat_node)
builder.add_node('translate', translate_node)

builder.add_edge(START, 'intent')
builder.add_conditional_edges('intent', intent_router)
builder.add_edge('weather', END)
builder.add_edge('chat', END)
builder.add_edge('translate', END)

graph= builder.compile()
# 用 xray 模式可以看到子图内部结构
print("\n=== xray=True模式（展开子图内部）===")
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))